# MeshGraphNet vortex-shedding depth study on a Kaggle GPU

Launcher for `physicsnemo-meshgraphnet-vortex`. It checks out one commit of the repository and calls its scripts; it contains no model or training code.

Session settings: **Accelerator** an NVIDIA GPU, **Internet** on.

| Step | What runs | When |
|---|---|---|
| A | environment, tests, dataset subset, CUDA smoke study, timing and memory probe | always |
| B | fixed-budget reduced study, `configs/full.yaml` (processor sizes 5, 10, 15; 3 seeds; 119,600 gradient steps each) | `RUN_FULL = True` |
| C | NVIDIA's settings, `configs/official.yaml` (processor size 15; 2,990,000 gradient steps) | `RUN_OFFICIAL = True` |

Before B and before C the notebook projects the runtime from a probe and compares it with what is left of `SESSION_BUDGET_HOURS`. If the projection does not fit, the step is **not started and nothing is reduced**: no epochs, seeds, trajectories or model sizes are changed. The notebook reports the numbers and finishes, and the study has to be redesigned explicitly in the repository.

In [ ]:
REPO_URL = "https://github.com/AdebanjiAdelowo/physicsnemo-meshgraphnet-vortex.git"
REF = "SET_TO_FULL_40_CHARACTER_COMMIT_SHA"   # output of `git rev-parse HEAD` for the commit to run
RUN_FULL = False                              # step B: only the value True starts it
RUN_OFFICIAL = False                          # step C: only the value True starts it
SESSION_BUDGET_HOURS = 9.0                    # GPU time you are willing to spend in this session; check Kaggle's current limits
GITHUB_TOKEN_SECRET = None                    # name of a Kaggle secret holding a GitHub token; only for a private repository

import re, time
T0 = time.time()
assert re.fullmatch(r"[0-9a-f]{40}", REF), "REF must be a full 40-character commit SHA, not a branch or tag."
WORK = "/kaggle/working"
REPO = f"{WORK}/physicsnemo-meshgraphnet-vortex"

def hours_left():
    return SESSION_BUDGET_HOURS - (time.time() - T0) / 3600

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "CUDA is not available. Select a GPU accelerator; this notebook does not fall back to CPU."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

In [ ]:
import os, subprocess
def sh(cmd, cwd=None, check=True):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True, check=check, cwd=cwd).returncode

def tree_is_clean():
    return not subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True).stdout.strip()

if not os.path.isdir(REPO):
    url = REPO_URL
    if GITHUB_TOKEN_SECRET:
        from kaggle_secrets import UserSecretsClient
        url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET) + "@")
    subprocess.run(["git", "clone", "--quiet", url, REPO], check=True)   # not echoed: the URL may hold a token
    subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--quiet", REF], cwd=REPO, check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
assert head == REF, f"checked out {head}, expected {REF}"
assert tree_is_clean(), "the working tree is not clean"
print("commit", head)

## Environment

PhysicsNeMo is pinned to a commit of its main branch and needs `torch>=2.13`; `pip` may replace the PyTorch build of the image. The PyG wheel index has no `torch_scatter` wheel for that PyTorch, so it is built against the installed PyTorch after everything else (CPU-only build: MeshGraphNet uses the sum aggregation, which `torch_scatter` implements with `torch.scatter_add_` and which runs on CUDA tensors). The last command runs in a new process, prints every version and checks the aggregation on the GPU.

In [ ]:
sh(f"grep -v '^torch_scatter' requirements.txt > {WORK}/requirements-no-scatter.txt", cwd=REPO)
sh(f"pip install -q -r {WORK}/requirements-no-scatter.txt", cwd=REPO)
sh("pip install -q setuptools wheel && FORCE_ONLY_CPU=1 pip install -q torch_scatter --no-build-isolation", cwd=REPO)
sh('python -c "import torch, physicsnemo, torch_geometric, torch_scatter, tfrecord, sys; assert torch.cuda.is_available(); '
   "out = torch_scatter.scatter(torch.ones(4, 2, device='cuda'), torch.tensor([0, 0, 1, 1], device='cuda'), dim=0, dim_size=2, reduce='sum'); "
   "assert out.is_cuda and out.tolist() == [[2.0, 2.0], [2.0, 2.0]]; "
   "print('python', sys.version.split()[0], '| torch', torch.__version__, '| CUDA', torch.version.cuda, '| physicsnemo', physicsnemo.__version__, "
   "'| torch_geometric', torch_geometric.__version__, '| torch_scatter', torch_scatter.__version__, '|', torch.cuda.get_device_name(0))\"", cwd=REPO)
assert tree_is_clean(), "the installation modified the clone"

In [ ]:
sh("python -m pytest -q", cwd=REPO)

## Dataset subset

The first 200 training, 10 validation and 50 test trajectories of DeepMind's `cylinder_flow` files (about 3.5 GB of the 16.4 GB). The script prints the expected size before it downloads, and `manifest.json` records the byte counts and SHA-256 of what was written. The training scripts refuse to start if a file holds fewer trajectories than the configuration needs.

In [ ]:
sh("python scripts/download_data.py --train 200 --valid 10 --test 50 --dry-run", cwd=REPO)
sh("python scripts/download_data.py --train 200 --valid 10 --test 50", cwd=REPO)
sh("cat data/cylinder_flow/manifest.json && df -h /kaggle/working | tail -1", cwd=REPO)

## CUDA smoke study

Two tiny models, 38 gradient steps. The numbers are not results.

In [ ]:
sh("python scripts/train.py --config smoke device=cuda name=smoke_cuda", cwd=REPO)
sh("python scripts/plot_results.py --study runs/smoke_cuda --out runs/smoke_cuda/figures", cwd=REPO)

## Timing and memory probe for step B

Trains every processor size of `configs/full.yaml` for 300 gradient steps and times one 599-step rollout. The projection extrapolates the measured steps per second to the 9 runs, with a 15 % allowance. Nothing of the study is started here.

In [ ]:
import json
probe_code = sh(f"python scripts/probe.py --config full --budget-hours {hours_left():.3f}", cwd=REPO, check=False)
assert probe_code in (0, 3), "the probe failed"
probe = json.load(open(f"{REPO}/runs/full_probe.json"))
for row in probe["per_processor_size"]:
    print(f"processor size {row['processor_size']:>2}: {row['steps_per_second']:.1f} steps/s, peak memory {row['peak_train_memory_mb']:.0f} MB, "
          f"{row['projected_train_hours_per_run']:.2f} h per run")
full_projection_h = probe["projected_hours_total"]
print(f"PROJECTION for step B: {full_projection_h:.2f} h ({probe['projected_hours_per_seed']:.2f} h per seed) on {probe['environment']['gpu_name']}. "
      f"Left of the session budget: {hours_left():.2f} h.")
sh(f"cp runs/full_probe.json {WORK}/physicsnemo-meshgraphnet-vortex-full-probe.json", cwd=REPO)

## Step B: fixed-budget reduced study (`configs/full.yaml`)

Runs are ordered by seed: all three processor sizes are trained for seed 0 before seed 1 starts, so an interrupted session still leaves complete seeds, and `--resume` continues from the finished runs.

In [ ]:
full_done = False
if RUN_FULL is not True:
    print("Step B skipped: RUN_FULL is False.")
elif full_projection_h > hours_left():
    print(f"STEP B NOT STARTED: projected {full_projection_h:.2f} h, {hours_left():.2f} h left of SESSION_BUDGET_HOURS = {SESSION_BUDGET_HOURS}.")
    print(f"The configuration was not reduced. Complete seeds that would fit: {probe['complete_seeds_within_budget']}.")
    print("Decide how to proceed before changing configs/full.yaml.")
else:
    sh("python scripts/train.py --config full --resume", cwd=REPO)
    sh("python scripts/plot_results.py --study runs/full --out runs/full/figures", cwd=REPO)
    sh("python scripts/results.py package runs/full", cwd=REPO)
    sh("python scripts/results.py bundle runs/full", cwd=REPO)
    sh(f"cp runs/full.zip {WORK}/physicsnemo-meshgraphnet-vortex-full.zip && cp runs/full_checkpoints.zip {WORK}/physicsnemo-meshgraphnet-vortex-full-checkpoints.zip", cwd=REPO)
    full_done = True
    print("STEP B COMPLETE.")

## Step C: NVIDIA's settings (`configs/official.yaml`)

400 training trajectories, 25 epochs, processor size 15: 2,990,000 gradient steps on one GPU. NVIDIA reports 8 A100 GPUs for this configuration. It runs only after step B has completed in this session and only if its own probe fits the remaining budget, which is not expected on a single Kaggle GPU.

In [ ]:
official_done = False
if RUN_OFFICIAL is not True:
    print("Step C skipped: RUN_OFFICIAL is False.")
elif not full_done:
    print("STEP C NOT STARTED: step B did not complete in this session, and it has priority.")
else:
    sh("python scripts/download_data.py --train 400 --valid 10 --test 50", cwd=REPO)
    official_code = sh(f"python scripts/probe.py --config official --budget-hours {hours_left():.3f}", cwd=REPO, check=False)
    assert official_code in (0, 3), "the probe failed"
    official_probe = json.load(open(f"{REPO}/runs/official_probe.json"))
    print(f"PROJECTION for step C: {official_probe['projected_hours_total']:.1f} h. Left of the session budget: {hours_left():.2f} h.")
    if official_probe["projected_hours_total"] > hours_left():
        print("STEP C NOT STARTED: it does not fit in the remaining budget. The configuration was not reduced.")
    else:
        sh("python scripts/train.py --config official --resume", cwd=REPO)
        sh("python scripts/plot_results.py --study runs/official --out runs/official/figures", cwd=REPO)
        sh("python scripts/results.py package runs/official", cwd=REPO)
        sh("python scripts/results.py bundle runs/official", cwd=REPO)
        sh(f"cp runs/official.zip {WORK}/physicsnemo-meshgraphnet-vortex-official.zip && cp runs/official_checkpoints.zip {WORK}/physicsnemo-meshgraphnet-vortex-official-checkpoints.zip", cwd=REPO)
        official_done = True
        print("STEP C COMPLETE.")

In [ ]:
import glob
assert tree_is_clean(), "the run modified tracked files of the clone"
print("Files in /kaggle/working:")
for path in sorted(glob.glob(f"{WORK}/physicsnemo-meshgraphnet-vortex-*")):
    if os.path.isfile(path):
        print(f"  {path}  {os.path.getsize(path) / 2**20:.1f} MB")
print(f"Elapsed: {(time.time() - T0) / 3600:.2f} h. Step A: done. Step B: {'done' if full_done else 'not run'}. Step C: {'done' if official_done else 'not run'}.")
if not full_done:
    print("KAGGLE CUDA CHECKS COMPLETE. No full study was run.")